# Genie Code Maturity Assessment - Config Reader

## Purpose
This notebook reads the completed SDLC Assessment CSV template and generates comprehensive maturity reports.

## Workflow
1. **Load Configuration** - Read the CSV file with client-provided scores
2. **Validate Data** - Ensure all required fields are completed
3. **Calculate Scores** - Compute overall and phase-specific maturity scores
4. **Identify Gaps** - Analyze gaps and prioritize improvements
5. **Generate Reports** - Create executive summary and detailed reports
6. **Visualize Results** - Build radar charts and heatmaps
7. **Export Results** - Save reports to Unity Catalog and generate PDFs

## Prerequisites
* Completed CSV file: `SDLC_Assessment_Template.csv`
* All questions scored (1-5 scale)
* Evidence and gap descriptions documented

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*100)
print("STEP 1: LOAD ASSESSMENT CONFIGURATION")
print("="*100)

# Configuration
CSV_PATH = '/Workspace/Users/sushant.mishriko@tigeranalytics.com/Genie Assessment Framework/SDLC_Assessment_Template.csv'

# Load the CSV file
df_assessment = pd.read_csv(CSV_PATH)

print(f"\n✅ Assessment file loaded successfully!")
print(f"\n📊 Total Questions: {len(df_assessment)}")
print(f"\n📋 Columns in DataFrame:")
for col in df_assessment.columns:
    print(f"  • {col}")

print(f"\n📝 Questions by Phase:")
for phase in df_assessment['Phase'].unique():
    count = len(df_assessment[df_assessment['Phase'] == phase])
    print(f"  • {phase}: {count} questions")

print("\n" + "="*100)
display(df_assessment.head(3))

In [0]:
print("="*100)
print("STEP 2: VALIDATE ASSESSMENT DATA")
print("="*100)

# Check for missing scores
missing_scores = df_assessment[df_assessment['Current_Score (1-5)'].isna() | (df_assessment['Current_Score (1-5)'] == '')]
print(f"\n📊 Validation Results:")
print(f"  • Total questions: {len(df_assessment)}")
print(f"  • Questions with scores: {len(df_assessment) - len(missing_scores)}")
print(f"  • Questions missing scores: {len(missing_scores)}")

if len(missing_scores) > 0:
    print("\n⚠️ WARNING: The following questions are missing scores:")
    for idx, row in missing_scores.iterrows():
        print(f"  • {row['Question_ID']}: {row['Question'][:70]}...")
    print("\n❌ Please complete all scores before proceeding.")
else:
    print("\n✅ All questions have been scored!")

# Convert Current_Score to numeric
df_assessment['Current_Score (1-5)'] = pd.to_numeric(df_assessment['Current_Score (1-5)'], errors='coerce')

# Validate score range (1-5)
invalid_scores = df_assessment[
    (df_assessment['Current_Score (1-5)'] < 1) | 
    (df_assessment['Current_Score (1-5)'] > 5)
]

if len(invalid_scores) > 0:
    print("\n⚠️ WARNING: The following questions have invalid scores (must be 1-5):")
    for idx, row in invalid_scores.iterrows():
        print(f"  • {row['Question_ID']}: Score = {row['Current_Score (1-5)']}")
else:
    print("\n✅ All scores are within valid range (1-5)!")

print("\n" + "="*100)

In [0]:
print("="*100)
print("STEP 3: CALCULATE MATURITY SCORES")
print("="*100)

# Calculate overall score
overall_score = df_assessment['Current_Score (1-5)'].mean()

# Calculate phase scores
phase_scores = df_assessment.groupby('Phase')['Current_Score (1-5)'].agg(['mean', 'count']).round(2)
phase_scores.columns = ['Average_Score', 'Question_Count']

# Define maturity levels
def get_maturity_level(score):
    if score < 1.5:
        return "Level 1: Initial/Ad-hoc"
    elif score < 2.5:
        return "Level 2: Aware/Experimental"
    elif score < 3.5:
        return "Level 3: Defined/Structured"
    elif score < 4.5:
        return "Level 4: Managed/Optimized"
    else:
        return "Level 5: Innovative/Leading"

overall_maturity = get_maturity_level(overall_score)

print(f"\n📊 OVERALL MATURITY")
print(f"  • Average Score: {overall_score:.2f} / 5.00")
print(f"  • Maturity Level: {overall_maturity}")

print(f"\n📋 PHASE-SPECIFIC SCORES")
for phase, row in phase_scores.iterrows():
    maturity = get_maturity_level(row['Average_Score'])
    print(f"  • {phase}")
    print(f"    - Score: {row['Average_Score']:.2f} / 5.00")
    print(f"    - Maturity: {maturity}")
    print(f"    - Questions: {int(row['Question_Count'])}")
    print()

print("="*100)

# Store results for later use
assessment_results = {
    'overall_score': overall_score,
    'overall_maturity': overall_maturity,
    'phase_scores': phase_scores,
    'assessment_date': datetime.now().strftime('%Y-%m-%d')
}

display(phase_scores)

In [0]:
print("="*100)
print("STEP 4: IDENTIFY STRENGTHS AND GAPS")
print("="*100)

# Identify strengths (score >= 3)
strengths = df_assessment[df_assessment['Current_Score (1-5)'] >= 3.5].sort_values('Current_Score (1-5)', ascending=False)

# Identify critical gaps (score <= 2)
critical_gaps = df_assessment[df_assessment['Current_Score (1-5)'] <= 2].sort_values('Current_Score (1-5)')

# Identify medium priority gaps (score 2-3)
medium_gaps = df_assessment[
    (df_assessment['Current_Score (1-5)'] > 2) & 
    (df_assessment['Current_Score (1-5)'] < 3.5)
].sort_values('Current_Score (1-5)')

print(f"\n✅ STRENGTHS (Score >= 3.5): {len(strengths)} questions")
if len(strengths) > 0:
    print("\nTop Strengths:")
    for idx, row in strengths.head(5).iterrows():
        print(f"  • {row['Question_ID']} (Score: {row['Current_Score (1-5)']:.1f}): {row['Question'][:70]}...")
else:
    print("  No strengths identified yet - significant improvement opportunity!")

print(f"\n🔴 CRITICAL GAPS (Score <= 2): {len(critical_gaps)} questions")
if len(critical_gaps) > 0:
    print("\nMost Critical Gaps:")
    for idx, row in critical_gaps.iterrows():
        print(f"  • {row['Question_ID']} (Score: {row['Current_Score (1-5)']:.1f}): {row['Question'][:70]}...")
        if pd.notna(row['Gap_Description']) and str(row['Gap_Description']).strip():
            print(f"    Gap: {row['Gap_Description'][:80]}...")
else:
    print("  No critical gaps - excellent work!")

print(f"\n🟡 MEDIUM PRIORITY GAPS (Score 2-3.5): {len(medium_gaps)} questions")
if len(medium_gaps) > 0:
    print("\nTop Medium Priority Gaps:")
    for idx, row in medium_gaps.head(5).iterrows():
        print(f"  • {row['Question_ID']} (Score: {row['Current_Score (1-5)']:.1f}): {row['Question'][:70]}...")

print("\n" + "="*100)

# Create summary DataFrames
strengths_summary = strengths[['Phase', 'Question_ID', 'Question', 'Current_Score (1-5)', 'Genie_Capability']]
gaps_summary = pd.concat([
    critical_gaps[['Phase', 'Question_ID', 'Question', 'Current_Score (1-5)', 'Gap_Description', 'Priority (Critical/High/Medium/Low)']],
    medium_gaps[['Phase', 'Question_ID', 'Question', 'Current_Score (1-5)', 'Gap_Description', 'Priority (Critical/High/Medium/Low)']]
]).sort_values('Current_Score (1-5)')

In [0]:
print("="*100)
print("EXECUTIVE SUMMARY REPORT")
print("="*100)
print(f"\nAssessment Date: {assessment_results['assessment_date']}")
print(f"Total Questions Assessed: {len(df_assessment)}")
print(f"\n" + "="*100)

print(f"\n🎯 OVERALL MATURITY")
print(f"  • Current Score: {overall_score:.2f} / 5.00")
print(f"  • Maturity Level: {overall_maturity}")

print(f"\n📊 KEY FINDINGS")
print(f"  • {len(strengths)} areas of strength (score >= 3.5)")
print(f"  • {len(critical_gaps)} critical gaps requiring immediate attention (score <= 2)")
print(f"  • {len(medium_gaps)} areas for improvement (score 2-3.5)")

print(f"\n📈 PHASE MATURITY BREAKDOWN")
for phase, row in phase_scores.sort_values('Average_Score').iterrows():
    maturity = get_maturity_level(row['Average_Score'])
    bar_length = int(row['Average_Score'] * 10)
    bar = '█' * bar_length + '░' * (50 - bar_length)
    print(f"  {phase}")
    print(f"    [{bar}] {row['Average_Score']:.2f} - {maturity}")

print(f"\n🔍 TOP 3 PRIORITY PHASES (Lowest Scores)")
top_priority_phases = phase_scores.sort_values('Average_Score').head(3)
for idx, (phase, row) in enumerate(top_priority_phases.iterrows(), 1):
    print(f"  {idx}. {phase}: {row['Average_Score']:.2f}/5.00")
    phase_gaps = df_assessment[
        (df_assessment['Phase'] == phase) & 
        (df_assessment['Current_Score (1-5)'] <= 2)
    ]
    if len(phase_gaps) > 0:
        print(f"     Critical gaps: {len(phase_gaps)} questions")

print(f"\n💡 RECOMMENDATIONS")
if overall_score < 2.5:
    print("  • Focus on Foundation: Build basic Genie Code capabilities")
    print("  • Start with training and awareness programs")
    print("  • Establish executive sponsorship and governance")
    print("  • Target: Reach Level 3 (Defined) within 6-9 months")
elif overall_score < 3.5:
    print("  • Expand Usage: Increase adoption across all SDLC phases")
    print("  • Implement custom skills and organizational standards")
    print("  • Build monitoring and observability frameworks")
    print("  • Target: Reach Level 4 (Managed) within 6-12 months")
elif overall_score < 4.5:
    print("  • Optimize & Innovate: Develop custom AI agents")
    print("  • Implement predictive analytics and auto-remediation")
    print("  • Build comprehensive STTM automation")
    print("  • Target: Reach Level 5 (Leading) within 9-15 months")
else:
    print("  • Maintain Excellence: Continue innovation")
    print("  • Share best practices with Databricks community")
    print("  • Explore cutting-edge AI capabilities")
    print("  • Become a reference customer")

print("\n" + "="*100)

In [0]:
import matplotlib.pyplot as plt
import numpy as np

print("Creating Radar Chart for Phase Maturity Scores...")

# Prepare data for radar chart
phases = phase_scores.index.tolist()
scores = phase_scores['Average_Score'].tolist()

# Number of variables
num_vars = len(phases)

# Compute angle for each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
scores += scores[:1]  # Complete the circle
angles += angles[:1]

# Create figure
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

# Draw the chart
ax.plot(angles, scores, 'o-', linewidth=2, label='Current Score', color='#1f77b4')
ax.fill(angles, scores, alpha=0.25, color='#1f77b4')

# Add target level (Level 3 = 3.0)
target = [3.0] * (num_vars + 1)
ax.plot(angles, target, '--', linewidth=1.5, label='Target (Level 3)', color='green', alpha=0.7)

# Fix axis to go in the right order and start at 12 o'clock
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Set labels and formatting
ax.set_xticks(angles[:-1])
ax.set_xticklabels(phases, size=10)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1', '2', '3', '4', '5'], size=8)
ax.set_rlabel_position(0)

# Add gridlines
ax.grid(True, linestyle='--', alpha=0.7)

# Add title and legend
plt.title(f'Genie Code Maturity Assessment\nOverall Score: {overall_score:.2f}/5.00 - {overall_maturity}', 
          size=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.show()

print("✅ Radar chart created successfully!")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Creating Heatmap for Question-Level Scores...")

# Prepare data for heatmap
heatmap_data = df_assessment.pivot_table(
    values='Current_Score (1-5)',
    index='Question_ID',
    columns='Phase',
    fill_value=0
)

# Create figure
fig, ax = plt.subplots(figsize=(14, 12))

# Create heatmap
sns.heatmap(
    heatmap_data.T,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    center=3,
    vmin=1,
    vmax=5,
    cbar_kws={'label': 'Maturity Score (1-5)'},
    linewidths=0.5,
    ax=ax
)

plt.title('Genie Code Maturity Assessment - Detailed Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Question ID', fontsize=12, fontweight='bold')
plt.ylabel('SDLC Phase', fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

print("✅ Heatmap created successfully!")

In [0]:
print("="*100)
print("PRIORITIZED ACTION PLAN")
print("="*100)

# Combine score and priority to create action plan
action_items = []

for idx, row in df_assessment.iterrows():
    score = row['Current_Score (1-5)']
    priority = str(row['Priority (Critical/High/Medium/Low)']).upper() if pd.notna(row['Priority (Critical/High/Medium/Low)']) else 'MEDIUM'
    
    # Calculate urgency score (lower score + higher priority = more urgent)
    urgency_map = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1, '': 2}
    urgency = urgency_map.get(priority, 2) * (6 - score)  # Inverse of score
    
    action_items.append({
        'Phase': row['Phase'],
        'Question_ID': row['Question_ID'],
        'Question': row['Question'],
        'Current_Score': score,
        'Target_Score': min(score + 2, 5),  # Target is 2 levels higher or max 5
        'Priority': priority,
        'Urgency_Score': urgency,
        'Gap_Description': row['Gap_Description'],
        'Genie_Capability': row['Genie_Capability'],
        'Example_Prompt': row['Example_Prompt']
    })

action_plan_df = pd.DataFrame(action_items).sort_values('Urgency_Score', ascending=False)

print(f"\n🎯 TOP 10 PRIORITY ACTIONS\n")
for idx, row in action_plan_df.head(10).iterrows():
    print(f"{row['Question_ID']}: {row['Question'][:70]}...")
    print(f"  • Current Score: {row['Current_Score']:.1f} → Target: {row['Target_Score']:.1f}")
    print(f"  • Priority: {row['Priority']} | Phase: {row['Phase']}")
    if pd.notna(row['Gap_Description']) and str(row['Gap_Description']).strip():
        print(f"  • Gap: {row['Gap_Description'][:100]}...")
    print(f"  • Try: {row['Example_Prompt']}")
    print()

print("="*100)

# Save action plan
action_plan_df.to_csv(
    '/Workspace/Users/sushant.mishriko@tigeranalytics.com/Genie Assessment Framework/Action_Plan.csv',
    index=False
)

print("\n✅ Action plan saved to: Action_Plan.csv")
display(action_plan_df.head(15))

In [0]:
print("="*100)
print("STEP 9: EXPORT RESULTS SUMMARY")
print("="*100)

# Create comprehensive summary
summary_data = {
    'Assessment_Date': [assessment_results['assessment_date']],
    'Overall_Score': [overall_score],
    'Overall_Maturity': [overall_maturity],
    'Total_Questions': [len(df_assessment)],
    'Strengths_Count': [len(strengths)],
    'Critical_Gaps_Count': [len(critical_gaps)],
    'Medium_Gaps_Count': [len(medium_gaps)],
    'Top_Priority_Phase': [phase_scores.sort_values('Average_Score').index[0]],
    'Top_Strength_Phase': [phase_scores.sort_values('Average_Score', ascending=False).index[0]]
}

summary_df = pd.DataFrame(summary_data)

# Save summary
summary_path = '/Workspace/Users/sushant.mishriko@tigeranalytics.com/Genie Assessment Framework/Assessment_Summary.csv'
summary_df.to_csv(summary_path, index=False)

print(f"\n✅ Summary exported to: Assessment_Summary.csv")

# Save detailed results with scores
detailed_path = '/Workspace/Users/sushant.mishriko@tigeranalytics.com/Genie Assessment Framework/Assessment_Detailed_Results.csv'
df_assessment.to_csv(detailed_path, index=False)

print(f"✅ Detailed results exported to: Assessment_Detailed_Results.csv")

# Display summary
print(f"\n📊 ASSESSMENT SUMMARY")
for col in summary_df.columns:
    print(f"  • {col}: {summary_df[col].iloc[0]}")

print("\n" + "="*100)
print("\n🎉 ASSESSMENT COMPLETE!")
print("\nGenerated Files:")
print("  1. Assessment_Summary.csv - Executive summary metrics")
print("  2. Assessment_Detailed_Results.csv - Complete assessment with scores")
print("  3. Action_Plan.csv - Prioritized improvement actions")
print("\nNext Steps:")
print("  1. Review the executive summary and radar chart")
print("  2. Share results with stakeholders")
print("  3. Prioritize top 5-10 action items")
print("  4. Create implementation roadmap")
print("  5. Schedule follow-up assessment in 3-6 months")
print("="*100)

display(summary_df)